# Racialised Reviews – Exploratory Data Analysis

Two main analyses:
1. **Reviewer cross-dataset analysis** – reviewers who have a racialised review: how many other reviews do they have, and in which category (non-social / social / racialised)?
2. **Keyword analysis** – occurrence of specific slang/terminology within the racialised reviews.

**Data pipeline recap**
```
social_classified_text_reviews.parquet   (1 265 806 reviews total)
    ├── social = no  →  761 733 reviews
    └── social = yes →  504 072 reviews
                            └── racialised_classified_text_reviews.parquet
                                    ├── racialised = no  →  493 279
                                    └── racialised = yes →   10 792
                                              └── racialised_merged_data_01062026.parquet
                                                      (7 768 cleaned/merged reviews)
```


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import re
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.1f}".format)


def linkify(df, cols):
    cols = [cols] if isinstance(cols, str) else cols
    fmt = {col: lambda u: f'<a href="{u}" target="_blank">open</a>' for col in cols}
    return df.style.format(fmt)


DATA_DIR = Path("../data")


## 1 · Load datasets


In [ ]:
# Largest corpus: all text reviews that went through social classification
print("Loading social_classified_text_reviews…")
social = pd.read_parquet(DATA_DIR / "social_classified_text_reviews.parquet",
                         #columns=["review_id", "reviewer_id", "social"]
                         )
print(f"  {len(social):,} rows")

# The racialised subset: all social=yes reviews with racialised label
print("Loading racialised_classified_text_reviews…")
rac_cls = pd.read_parquet(DATA_DIR / "racialised_classified_text_reviews.parquet",
                          #columns=["review_id", "reviewer_id", "racialised"]
                          )
print(f"  {len(rac_cls):,} rows  |  racialised=yes: {(rac_cls['racialised']=='yes').sum():,}")

# The final merged/cleaned racialised dataset (with full labels & place metadata)
print("Loading racialised_merged_data_01062026…")
rac_merged = pd.read_parquet(DATA_DIR / "racialised_merged_data_01062026.parquet")
print(f"  {len(rac_merged):,} rows")

print("Loading places_filtered.parquet…")
places = pd.read_parquet(DATA_DIR / "places_filtered.parquet")
print(f"  {len(places):,} rows")

---
## 2 · Reviewer cross-dataset analysis

Starting from reviewers who appear in the **racialised merged dataset** (7 768 reviews),
we trace all of their reviews in the full social-classified corpus and categorise each as:
- **racialised** – review appears in `racialised_classified` with `racialised = yes`
- **social** – classified as `social = yes` but *not* racialised
- **non_social** – classified as `social = no`


In [ ]:
# Reviewer IDs who have at least one confirmed racialised review
racialised_reviewer_ids = set(rac_merged["reviewer_id"].dropna().unique())
print(f"Unique reviewers in racialised merged dataset: {len(racialised_reviewer_ids):,}")

# All reviews by those reviewers in the large corpus
corpus = social[social["reviewer_id"].isin(racialised_reviewer_ids)].copy()
print(f"Total reviews in the corpus by these reviewers: {len(corpus):,}")
print(f"Unique reviewers found in the larger corpus:    {corpus['reviewer_id'].nunique():,}")


In [ ]:
# Set of review_ids that are confirmed racialised (from classified file)
racialised_review_ids = set(rac_cls.loc[rac_cls["racialised"] == "yes", "review_id"])

# Categorise each review
def categorise(row):
    if row["review_id"] in racialised_review_ids:
        return "racialised"
    elif row["social"] == "yes":
        return "social"
    else:
        return "non_social"

corpus["category"] = corpus.apply(categorise, axis=1)

# Sanity check
print("Category breakdown across all reviews by these reviewers:")
print(corpus["category"].value_counts().to_frame("count")
      .assign(pct=lambda d: (d["count"] / d["count"].sum() * 100).round(1)))


In [ ]:
social.info()

In [ ]:
# Reviewers with multiple reviews anywhere in the corpus
review_counts = corpus.groupby("reviewer_id").size().rename("total_reviews")
multi_reviewers = review_counts[review_counts > 1]
print(f"Reviewers with >1 review in the corpus: {len(multi_reviewers):,} "
      f"({len(multi_reviewers)/len(racialised_reviewer_ids)*100:.1f}% of racialised reviewers)")
print()
print("Distribution of total reviews per reviewer:")
print(review_counts.describe().rename("reviews").to_frame().T.to_string())


In [ ]:
# Per-reviewer category breakdown
cat_pivot = (corpus.groupby(["reviewer_id", "category"])
             .size()
             .unstack(fill_value=0)
             .reindex(columns=["non_social", "social", "racialised"], fill_value=0))
cat_pivot["total"] = cat_pivot.sum(axis=1)

print(f"Reviewers appearing in the corpus: {len(cat_pivot):,}")
print()
print("Reviewers by how many reviews they have in each category:")
print(cat_pivot.describe().round(2))


In [ ]:
# How many reviewers have reviews in EACH category (not mutually exclusive)
for col in ["non_social", "social", "racialised"]:
    n = (cat_pivot[col] > 0).sum()
    print(f"Reviewers with ≥1 {col:<12} review: {n:>5,}  ({n/len(cat_pivot)*100:.1f}%)")


In [ ]:
# Reviewers who have reviews in MORE than one category
multi_cat = cat_pivot[(cat_pivot > 0).sum(axis=1) > 1]
print(f"Reviewers whose reviews span >1 category: {len(multi_cat):,}  "
      f"({len(multi_cat)/len(cat_pivot)*100:.1f}%)")
print()

# Combinations of categories
cat_pivot["has_non_social"] = (cat_pivot["non_social"] > 0).astype(int)
cat_pivot["has_social"]     = (cat_pivot["social"]     > 0).astype(int)
cat_pivot["has_racialised"] = (cat_pivot["racialised"] > 0).astype(int)

combo = cat_pivot.groupby(["has_non_social", "has_social", "has_racialised"]).size().rename("n_reviewers")
combo_df = combo.reset_index()
combo_df.columns = ["has_non_social", "has_social", "has_racialised", "n_reviewers"]
combo_df = combo_df.sort_values("n_reviewers", ascending=False)
print("Reviewer category combinations (1 = has at least one review in that category):")
print(combo_df.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Reviewer Cross-Dataset Analysis", fontsize=14, fontweight="bold")

# --- Plot 1: review count distribution (capped at 20 for readability) ---
ax = axes[0]
capped = review_counts.clip(upper=20)
counts = capped.value_counts().sort_index()
ax.bar(counts.index.astype(str), counts.values, color="#4C72B0", edgecolor="white")
ax.set_xlabel("Total reviews in corpus (20 = ≥20)")
ax.set_ylabel("Number of reviewers")
ax.set_title("Review count distribution\n(racialised reviewers in full corpus)")
ax.tick_params(axis="x", rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# --- Plot 2: category breakdown (total review counts) ---
ax = axes[1]
cat_totals = corpus["category"].value_counts().reindex(["non_social", "social", "racialised"])
colors = ["#55A868", "#4C72B0", "#C44E52"]
bars = ax.bar(cat_totals.index, cat_totals.values, color=colors, edgecolor="white")
ax.set_title("Total reviews by category\n(all reviews by racialised reviewers)")
ax.set_ylabel("Number of reviews")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
for bar, val in zip(bars, cat_totals.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f"{val:,}", ha="center", va="bottom", fontsize=9)

# --- Plot 3: reviewer category combinations ---
ax = axes[2]
combo_plot = combo_df.copy()
combo_plot["label"] = combo_plot.apply(
    lambda r: "/".join([k.replace("has_","") for k in ["has_non_social","has_social","has_racialised"] if r[k]==1]),
    axis=1)
combo_plot = combo_plot.sort_values("n_reviewers", ascending=True)
bars = ax.barh(combo_plot["label"], combo_plot["n_reviewers"], color="#DD8452", edgecolor="white")
ax.set_title("Reviewer category overlap\n(which category combinations do they have?)")
ax.set_xlabel("Number of reviewers")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
for bar, val in zip(bars, combo_plot["n_reviewers"]):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("../notebooks/visualisations/reviewer_cross_dataset.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Restrict to the 1,370 reviewers who have at least one review OUTSIDE the racialised dataset.
# The full describe() is misleading because 71% of reviewers appear only in the racialised set,
# so non_social and social columns are zero for most rows, pulling medians to 0.
active = cat_pivot[(cat_pivot["non_social"] > 0) | (cat_pivot["social"] > 0)].copy()

print(f"Reviewers with reviews outside the racialised dataset: {len(active):,} of {len(cat_pivot):,} ({len(active)/len(cat_pivot)*100:.1f}%)")
print()
print("Review composition for this group (counts per reviewer):")
display(active[["non_social", "social", "racialised", "total"]].describe().round(2))
print()
print("Median reviewer in this group has:")
for col, val in active[["non_social", "social", "racialised", "total"]].median().items():
    print(f"  {col:<12} {val:.0f} review(s)")

---
## 3 · Keyword analysis in racialised reviews

Searching in **both** `org_review` (original language) and `trans_review` (English translation).
Each keyword group counted **at most once per review**, regardless of how many times it appears.


In [ ]:
# Use the full merged racialised dataset for keyword search
rac_text = rac_merged[["review_id", "org_review", "trans_review"]].copy()
print(f"Searching across {len(rac_text):,} racialised reviews…")

KEYWORD_GROUPS = {
    "racist / racism / racially":            r"\bracis[mt]|\bracially\b|\bracism\b",
    "xenophobe / xenophobic / xenophobia":   r"\bxenophob",
    "Talahon / Talahun (+ plurals)":         r"\btalah[ou]n[s]?\b",
    "Kanacken / Kanack / Kanacks":           r"\bkanack[s]?\b|\bkanacken\b",
}

def keyword_hits(df, pattern):
    """Return boolean Series: True if pattern found in org_review OR trans_review."""
    regex = re.compile(pattern, re.IGNORECASE)
    mask = pd.Series(False, index=df.index)
    for col in ["org_review", "trans_review"]:
        mask |= df[col].fillna("").str.contains(regex, regex=True)
    return mask

results = {}
for label, pat in KEYWORD_GROUPS.items():
    hits = keyword_hits(rac_text, pat)
    results[label] = {
        "reviews_with_keyword": hits.sum(),
        "pct_of_racialised": round(hits.sum() / len(rac_text) * 100, 2),
    }

kw_df = pd.DataFrame(results).T.reset_index().rename(columns={"index": "keyword_group"})
kw_df["reviews_with_keyword"] = kw_df["reviews_with_keyword"].astype(int)
print(kw_df.to_string(index=False))


In [ ]:
# Also break down: where was the match found (org only, trans only, both)?
breakdown = []
for label, pat in KEYWORD_GROUPS.items():
    regex = re.compile(pat, re.IGNORECASE)
    in_org   = rac_text["org_review"].fillna("").str.contains(regex, regex=True)
    in_trans = rac_text["trans_review"].fillna("").str.contains(regex, regex=True)
    breakdown.append({
        "keyword_group":  label,
        "org_only":      int((in_org & ~in_trans).sum()),
        "trans_only":    int((~in_org & in_trans).sum()),
        "both":          int((in_org & in_trans).sum()),
        "total_reviews": int((in_org | in_trans).sum()),
    })

bk_df = pd.DataFrame(breakdown)
print("Match source breakdown (per-review, each counted once):")
bk_df


In [ ]:
# Show sample reviews for each keyword group
for label, pat in KEYWORD_GROUPS.items():
    hits_mask = keyword_hits(rac_text, pat)
    sample = rac_merged.loc[hits_mask, ["review_id", "org_review", "trans_review"]].head(3)
    print(f"\n{'='*70}")
    print(f"  {label}  ({hits_mask.sum()} reviews total)")
    print(f"{'='*70}")
    for _, row in sample.iterrows():
        print(f"  [org]   {str(row['org_review'])[:200]}")
        print(f"  [trans] {str(row['trans_review'])[:200]}")
        print()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Keyword Occurrence in Racialised Reviews", fontsize=14, fontweight="bold")

# --- Plot 1: absolute counts ---
ax = axes[0]
short_labels = [l.split("/")[0].strip() + "…" if "/" in l else l for l in kw_df["keyword_group"]]
bars = ax.barh(short_labels, kw_df["reviews_with_keyword"], color="#4C72B0", edgecolor="white")
ax.set_xlabel("Number of reviews containing keyword")
ax.set_title("Absolute keyword counts\n(each keyword counted once per review)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
for bar, val in zip(bars, kw_df["reviews_with_keyword"]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f"{int(val):,}", va="center", fontsize=10)

# --- Plot 2: stacked (org_only / trans_only / both) ---
ax = axes[1]
bk_plot = bk_df.set_index("keyword_group")[["org_only","trans_only","both"]]
bk_plot.index = [l.split("/")[0].strip() + "…" if "/" in l else l for l in bk_plot.index]
bk_plot.plot(kind="barh", stacked=True, ax=ax,
             color=["#4C72B0","#55A868","#C44E52"], edgecolor="white")
ax.set_xlabel("Number of reviews")
ax.set_title("Match source breakdown\n(org_review vs trans_review)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.legend(title="Found in", loc="lower right")

plt.tight_layout()
plt.savefig("../notebooks/visualisations/keyword_analysis.png", dpi=150, bbox_inches="tight")
plt.show()


---
*Notebook generated 2026-06-30 – swimming pool project EDA*


---
## 4 · Reviewers with multiple racialised reviews

121 reviewers appear more than once in the racialised merged dataset. We examine:
- Are they reviewing the **same pool** or different pools?
- Are they **geographically clustered** (same state) or travelling?
- Are their review **texts similar** (copy-paste behaviour) or genuinely different?


In [ ]:
from difflib import SequenceMatcher

rac_c = rac_merged.groupby('reviewer_id').size()
repeat_ids = rac_c[rac_c > 1].index
repeat_df = rac_merged[rac_merged['reviewer_id'].isin(repeat_ids)].sort_values(['reviewer_id','review_time']).copy()

print('Repeat reviewers:', len(repeat_ids))
print('Their total racialised reviews:', len(repeat_df))
print()
print('Review count per reviewer:')
print(rac_c[rac_c > 1].value_counts().rename('n_reviewers').to_frame().rename_axis('racialised_reviews'))


In [ ]:
repeat_df['postcode'] = repeat_df['address'].str.extract(r'\b(\d{5})\b')
repeat_df['city']     = repeat_df['address'].str.extract(r'\d{5}\s+([^,]+)')

def plz_to_state(plz):
    if pd.isna(plz): return 'Unknown'
    p = int(plz[:2])
    if p < 10:  return 'Sachsen/Thüringen/SA'
    if p < 17:  return 'Berlin/Brandenburg'
    if p < 20:  return 'Mecklenburg-VP'
    if p < 26:  return 'Hamburg/SH'
    if p < 32:  return 'Niedersachsen/Bremen'
    if p < 40:  return 'NRW/Niedersachsen'
    if p < 50:  return 'Nordrhein-Westfalen'
    if p < 56:  return 'NRW/Hessen/RP'
    if p < 60:  return 'Hessen/RP'
    if p < 70:  return 'Hessen/BW/RP'
    if p < 80:  return 'Baden-Württemberg'
    if p < 90:  return 'Bayern'
    return 'Bayern/Thüringen'

repeat_df['state_approx'] = repeat_df['postcode'].apply(plz_to_state)

reviewer_geo = repeat_df.groupby('reviewer_id').agg(
    reviewer_name = ('reviewer_name', 'first'),
    n_reviews     = ('review_id', 'count'),
    n_pools       = ('place_id', 'nunique'),
    n_states      = ('state_approx', 'nunique'),
    states        = ('state_approx', lambda x: ' | '.join(sorted(x.dropna().unique()))),
    cities        = ('city',         lambda x: ' | '.join(sorted(x.dropna().unique()))),
    pools         = ('place_name',   lambda x: ' | '.join(sorted(x.dropna().astype(str).unique()))),
)

print('Same vs different pools:')
print(reviewer_geo['n_pools'].value_counts().rename('n_reviewers').to_frame().rename_axis('distinct_pools'))
print()
print('State spread (distinct state regions visited):')
print(reviewer_geo['n_states'].value_counts().rename('n_reviewers').to_frame().rename_axis('distinct_state_regions'))


In [ ]:
same_state_mask = reviewer_geo['n_states'] == 1
same_state_reviewers = reviewer_geo[same_state_mask].index
state_counts = (
    repeat_df[repeat_df['reviewer_id'].isin(same_state_reviewers)]
    .groupby('state_approx')['reviewer_id'].nunique()
    .sort_values(ascending=False)
)
print('Reviewers staying within ONE state region:', same_state_mask.sum(), '/', len(reviewer_geo))
print()
print('Which state region?')
print(state_counts.rename('n_reviewers'))
print()
print('Reviewers crossing into 2 state regions:', (reviewer_geo['n_states']==2).sum())
print('Reviewers crossing into 3 state regions:', (reviewer_geo['n_states']==3).sum())


In [ ]:
sim_results = []
for rid, grp in repeat_df.groupby('reviewer_id'):
    texts = grp['trans_review'].fillna('').tolist()
    max_sim = 0.0
    for i in range(len(texts)):
        for j in range(i+1, len(texts)):
            r = SequenceMatcher(None, texts[i], texts[j]).ratio()
            if r > max_sim:
                max_sim = r
    sim_results.append({'reviewer_id': rid, 'max_text_similarity': round(max_sim, 3)})

sim_df = pd.DataFrame(sim_results).set_index('reviewer_id')
reviewer_geo = reviewer_geo.join(sim_df)

bins   = [0, 0.2, 0.4, 0.6, 0.8, 0.9, 1.01]
labels = ['0-0.2 (unique)', '0.2-0.4', '0.4-0.6', '0.6-0.8',
          '0.8-0.9 (very similar)', '0.9-1.0 (near/exact duplicate)']
reviewer_geo['sim_bin'] = pd.cut(reviewer_geo['max_text_similarity'], bins=bins, labels=labels, right=False)

print('Text similarity distribution:')
print(reviewer_geo['sim_bin'].value_counts().sort_index().rename('n_reviewers'))


In [ ]:
near_dup = reviewer_geo[reviewer_geo['max_text_similarity'] >= 0.9]
print('Near-duplicate reviewers (similarity >= 0.9):', len(near_dup))
print(near_dup[['reviewer_name','n_reviews','n_pools','states','max_text_similarity']].to_string())
print()
for rid, row in near_dup.iterrows():
    revs = repeat_df[repeat_df['reviewer_id'] == rid][['place_name','city','trans_review']]
    print('='*70)
    print(' ', row['reviewer_name'], '  similarity=' + str(row['max_text_similarity']))
    for _, r in revs.iterrows():
        print('  [' + str(r['place_name']) + ' | ' + str(r['city']) + ']')
        print(' ', str(r['trans_review'])[:350])
        print()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Repeat Reviewers in Racialised Dataset (n=121)', fontsize=13, fontweight='bold')

ax = axes[0]
pool_counts = reviewer_geo['n_pools'].value_counts().sort_index()
bars = ax.bar(pool_counts.index.astype(str), pool_counts.values, color='#4C72B0', edgecolor='white')
ax.set_xlabel('Distinct pools reviewed')
ax.set_ylabel('Number of reviewers')
ax.set_title('How many different pools\ndid each repeat reviewer visit?')
for bar, v in zip(bars, pool_counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, str(v), ha='center', fontsize=9)

ax = axes[1]
state_spread = reviewer_geo['n_states'].value_counts().sort_index()
bars2 = ax.bar([str(x) for x in state_spread.index], state_spread.values,
               color=['#55A868','#DD8452','#C44E52'][:len(state_spread)], edgecolor='white')
ax.set_xlabel('Distinct state regions visited')
ax.set_ylabel('Number of reviewers')
ax.set_title('Geographic spread\n(approximate state regions)')
for bar, v in zip(bars2, state_spread.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, str(v), ha='center', fontsize=9)

ax = axes[2]
sim_counts = reviewer_geo['sim_bin'].value_counts().sort_index()
short_labels = ['0-0.2\nunique','0.2-0.4','0.4-0.6','0.6-0.8','0.8-0.9\nv.similar','0.9-1.0\nnear-dup']
bars3 = ax.bar(short_labels[:len(sim_counts)], sim_counts.values, color='#8172B2', edgecolor='white')
ax.set_xlabel('Max pairwise text similarity')
ax.set_ylabel('Number of reviewers')
ax.set_title('Review text similarity\n(are they copy-pasting?)')
for bar, v in zip(bars3, sim_counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, str(v), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../notebooks/visualisations/repeat_reviewer_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
display_cols = ['reviewer_name','n_reviews','n_pools','n_states','states','cities','max_text_similarity']
reviewer_geo[display_cols].sort_values('n_reviews', ascending=False)


### Note: the one same-pool reviewer-pool pair — Thomas Kaufmann at Neckarbad

The dataset has a single reviewer-pool pair with two entries. Both share the same `reviewer_id` and `place_id` but have **different `review_id`s**. This is a data artefact: Google assigned a new ID when the review was heavily edited, and both versions were captured at different scraping runs.

In [ ]:
tk = rac_merged[
    (rac_merged["reviewer_id"] == "101393781172042826612") &
    (rac_merged["place_id"]    == "ChIJdduAUvWXkEcRfgEGCKrfObw")
][["review_id", "reviewer_name", "place_name", "review_time", "org_review", "trans_review"]].reset_index(drop=True)

for i, row in tk.iterrows():
    print(f"{'='*80}")
    print(f"Review {i+1} of 2  |  review_time: {row['review_time']}")
    print(f"review_id: {row['review_id']}")
    print(f"{'─'*80}")
    print(f"[DE] {row['org_review']}")
    print()
    print(f"[EN] {row['trans_review']}")
    print()

---
## 4b · Broader corpus activity: all 7,627 reviewers vs the 121 repeat racialised reviewers

Do racialised reviewers have other reviews in the larger corpus — and does having multiple
racialised reviews predict more activity overall?

Three groups compared:
- **All** 7,627 unique racialised reviewers
- **121** with more than one racialised review
- **7,506** with exactly one racialised review

In [ ]:
rac_counts   = rac_merged.groupby("reviewer_id").size()
single_ids   = set(rac_counts[rac_counts == 1].index)
repeat_ids_r = set(rac_counts[rac_counts > 1].index)
all_ids      = set(rac_merged["reviewer_id"].dropna())

racialised_review_ids2 = set(rac_cls.loc[rac_cls["racialised"] == "yes", "review_id"])

def corpus_pivot(reviewer_ids):
    sub = social[social["reviewer_id"].isin(reviewer_ids)].copy()
    def cat(row):
        if row["review_id"] in racialised_review_ids2: return "racialised"
        elif row["social"] == "yes":                   return "social"
        else:                                          return "non_social"
    sub["category"] = sub.apply(cat, axis=1)
    pivot = (sub.groupby(["reviewer_id", "category"]).size()
               .unstack(fill_value=0)
               .reindex(columns=["non_social", "social", "racialised"], fill_value=0))
    pivot["total"] = pivot.sum(axis=1)
    return pivot

piv_all    = corpus_pivot(all_ids)
piv_repeat = corpus_pivot(repeat_ids_r)
piv_single = corpus_pivot(single_ids)

def summary_row(label, n_group, pivot):
    has_other = ((pivot["non_social"] > 0) | (pivot["social"] > 0)).sum()
    return {
        "Group":                        label,
        "Reviewers":                    n_group,
        "In corpus":                    len(pivot),
        "Only racialised reviews":      (pivot["total"] == pivot["racialised"]).sum(),
        "Have other reviews":           has_other,
        "% with other reviews":         f"{has_other / n_group * 100:.1f}%",
        "Have ≥1 non_social":           (pivot["non_social"] > 0).sum(),
        "% with non_social":            f"{(pivot['non_social']>0).sum() / n_group * 100:.1f}%",
        "Have ≥1 social (not rac)":     (pivot["social"] > 0).sum(),
        "% with social":                f"{(pivot['social']>0).sum() / n_group * 100:.1f}%",
    }

rows = [
    summary_row("All 7,627",            len(all_ids),    piv_all),
    summary_row("121 (repeat rac.)",    len(repeat_ids_r), piv_repeat),
    summary_row("7,506 (single rac.)",  len(single_ids), piv_single),
]
summary_tbl = pd.DataFrame(rows).set_index("Group")
print("Corpus activity summary by reviewer group:")
display(summary_tbl)

In [ ]:
print("Review composition — reviewers WITH other corpus activity (per reviewer stats):")
print()
for label, pivot in [("All 7,627", piv_all), ("121 repeat", piv_repeat), ("7,506 single", piv_single)]:
    active = pivot[(pivot["non_social"] > 0) | (pivot["social"] > 0)]
    print(f"  {label}  (n={len(active):,} with other reviews)")
    desc = active[["non_social", "social", "racialised", "total"]].describe().loc[["mean","50%","max"]].round(2)
    desc.index = ["mean", "median", "max"]
    print(desc.to_string())
    print()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Broader Corpus Activity by Reviewer Group", fontsize=13, fontweight="bold")

groups = {
    "All 7,627":     (piv_all,    len(all_ids)),
    "121 (repeat)":  (piv_repeat, len(repeat_ids_r)),
    "7,506 (single)":(piv_single, len(single_ids)),
}

# Plot 1: % with / without other reviews
ax = axes[0]
labels_g  = list(groups.keys())
pct_other = [((p["non_social"]>0)|(p["social"]>0)).sum()/n*100 for p, n in groups.values()]
pct_only  = [100 - p for p in pct_other]
x = range(len(labels_g))
ax.bar(x, pct_only,  label="Racialised only", color="#4C72B0", edgecolor="white")
ax.bar(x, pct_other, bottom=pct_only, label="Has other reviews", color="#DD8452", edgecolor="white")
ax.set_xticks(x); ax.set_xticklabels(labels_g, fontsize=9)
ax.set_ylabel("% of reviewers"); ax.set_ylim(0, 110)
ax.set_title("Share with other corpus activity")
ax.legend(fontsize=8)
for xi, (po, pa) in enumerate(zip(pct_only, pct_other)):
    ax.text(xi, po/2,       f"{po:.1f}%", ha="center", va="center", fontsize=8, color="white", fontweight="bold")
    ax.text(xi, po + pa/2,  f"{pa:.1f}%", ha="center", va="center", fontsize=8, color="white", fontweight="bold")

# Plot 2: % with >=1 non_social vs >=1 social
ax = axes[1]
pct_ns = [(p["non_social"]>0).sum()/n*100 for p, n in groups.values()]
pct_s  = [(p["social"]>0).sum()/n*100      for p, n in groups.values()]
w = 0.35
xi = [i - w/2 for i in range(3)]
xi2= [i + w/2 for i in range(3)]
b1 = ax.bar(xi,  pct_ns, w, label="≥1 non-social",        color="#55A868", edgecolor="white")
b2 = ax.bar(xi2, pct_s,  w, label="≥1 social (not rac.)", color="#C44E52", edgecolor="white")
ax.set_xticks(range(3)); ax.set_xticklabels(labels_g, fontsize=9)
ax.set_ylabel("% of reviewers"); ax.set_title("Type of other reviews present")
ax.legend(fontsize=8)
for bar, v in list(zip(b1, pct_ns)) + list(zip(b2, pct_s)):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f"{v:.1f}%", ha="center", fontsize=7.5)

# Plot 3: median total reviews (among those with other activity)
ax = axes[2]
medians = []
for label, (pivot, _) in groups.items():
    active = pivot[(pivot["non_social"]>0)|(pivot["social"]>0)]
    medians.append({
        "group":      label,
        "non_social": active["non_social"].median(),
        "social":     active["social"].median(),
        "racialised": active["racialised"].median(),
    })
med_df = pd.DataFrame(medians).set_index("group")
med_df.plot(kind="bar", ax=ax, color=["#55A868","#C44E52","#4C72B0"], edgecolor="white", rot=15)
ax.set_ylabel("Median review count per reviewer")
ax.set_title("Median reviews per category\n(reviewers with other activity only)")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("../notebooks/visualisations/corpus_activity_by_group.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 5 · Distribution of racialised reviews across pools, constituencies and states

**1,623 distinct pools** have at least one racialised review across **7,768 reviews total**.
All pools are spatially matched to the 299 federal-election constituencies (Wahlkreise)
using a point-in-polygon join on the official BTW25 shapefile.


In [ ]:
import geopandas as gpd
from shapely.geometry import Point

# Pool-level aggregation
pool_df = (
    rac_merged
    .groupby(['place_id','place_name','address','latitude','longitude'])
    .size()
    .reset_index(name='racialised_count')
)

print('Distinct pools with >=1 racialised review:', len(pool_df))
print()
print('Reviews per pool:')
print(pool_df['racialised_count'].describe().round(2))
print()
print('Pool count breakdown:')
bins = [1,2,3,4,5,10,20,50,999]
labels = ['1','2','3','4','5-9','10-19','20-49','50+']
pool_df['count_bin'] = pd.cut(pool_df['racialised_count'], bins=bins, labels=labels, right=False)
print(pool_df['count_bin'].value_counts().sort_index().rename('pools'))


In [ ]:
print('Top 20 pools by racialised review count:')
top_pools = pool_df.nlargest(20, 'racialised_count')[['place_name','address','racialised_count']]
top_pools


In [ ]:
places= pd.read_parquet('/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/places_filtered.parquet')
places[places['place_name']=='Tropical Islands']

In [ ]:
print(places[places['place_name']=='Tropical Islands']['google_url'])

In [ ]:
linkify(places[places['place_name']=='Tropical Islands'], 'google_url')

In [ ]:
social.groupby('unique_identifier')['total_text_reviews'].count().sort_values(ascending=False).head(20)

In [ ]:
social.info()

In [ ]:
social[social['unique_identifier']=='0x47a804edb23f5915:0x20148692752fbd91']

In [ ]:
 social[social['unique_identifier']=='0x47a804edb23f5915:0x20148692752fbd91']['review_time'].unique()

In [ ]:
social[social['unique_identifier']=='0x47a804edb23f5915:0x20148692752fbd91']['review_time'], social[social['unique_identifier']=='0x47a804edb23f5915:0x20148692752fbd91'].groupby('review_time').size().sort_values(ascending=False)

In [ ]:
linkify(social[social['unique_identifier']=='0x47bf31d7722b5eb9:0xc57e8ba57033529a'][['review_id','review_time','trans_review', 'retrieved_time','contributor_link', 'review_link']].head(20), ['contributor_link', 'review_link'])

In [ ]:
rac_merged.info()    

In [ ]:
WK_PATH = Path('/home/ethel/Desktop/swimming_pool_project/btw25_geometrie_wahlkreise_shp'
               '/btw25_geometrie_wahlkreise_shp.shp')
wk = gpd.read_file(WK_PATH)

pool_gdf = gpd.GeoDataFrame(
    pool_df,
    geometry=[Point(lon, lat) for lon, lat in zip(pool_df['longitude'], pool_df['latitude'])],
    crs='EPSG:4326'
).to_crs(wk.crs)

pool_joined = gpd.sjoin(
    pool_gdf,
    wk[['WKR_NR','WKR_NAME','LAND_NAME','geometry']],
    how='left', predicate='within'
)

print('Pools matched to a constituency:', pool_joined['WKR_NR'].notna().sum(), '/', len(pool_joined))


In [ ]:
state_df = (
    pool_joined
    .groupby('LAND_NAME')
    .agg(
        n_pools           = ('place_id', 'count'),
        total_racialised  = ('racialised_count', 'sum'),
        median_per_pool   = ('racialised_count', 'median'),
        max_per_pool      = ('racialised_count', 'max'),
    )
    .sort_values('total_racialised', ascending=False)
    .reset_index()
)
state_df['reviews_per_pool'] = (state_df['total_racialised'] / state_df['n_pools']).round(2)
print('State-level summary:')
state_df


In [ ]:
const_df = (
    pool_joined
    .groupby(['WKR_NR','WKR_NAME','LAND_NAME'])
    .agg(
        n_pools          = ('place_id', 'count'),
        total_racialised = ('racialised_count', 'sum'),
    )
    .sort_values('total_racialised', ascending=False)
    .reset_index()
)
const_df['reviews_per_pool'] = (const_df['total_racialised'] / const_df['n_pools']).round(2)
print('Top 20 constituencies by total racialised reviews:')
const_df.head(20)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Racialised Reviews – Pool-Level Distribution', fontsize=13, fontweight='bold')

# histogram (log y)
ax = axes[0]
bin_counts = pool_df['count_bin'].value_counts().sort_index()
bars = ax.bar(bin_counts.index.astype(str), bin_counts.values, color='#4C72B0', edgecolor='white')
ax.set_yscale('log')
ax.set_xlabel('Racialised reviews per pool')
ax.set_ylabel('Number of pools (log scale)')
ax.set_title('How many pools have how many reviews?')
for bar, v in zip(bars, bin_counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.05, str(v), ha='center', fontsize=9)

# top-15 pools
ax = axes[1]
top15 = pool_df.nlargest(15, 'racialised_count').sort_values('racialised_count')
short_names = [n[:35]+'…' if len(str(n)) > 35 else str(n) for n in top15['place_name']]
bars2 = ax.barh(short_names, top15['racialised_count'], color='#C44E52', edgecolor='white')
ax.set_xlabel('Racialised reviews')
ax.set_title('Top 15 pools by racialised review count')
for bar, v in zip(bars2, top15['racialised_count']):
    ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2, str(v), va='center', fontsize=8)

plt.tight_layout()
plt.savefig('../notebooks/visualisations/pool_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Racialised Reviews by State (Bundesland)', fontsize=13, fontweight='bold')

state_sorted = state_df.sort_values('total_racialised', ascending=True)

ax = axes[0]
bars = ax.barh(state_sorted['LAND_NAME'], state_sorted['total_racialised'], color='#4C72B0', edgecolor='white')
ax.set_xlabel('Total racialised reviews')
ax.set_title('Total racialised reviews per state')
for bar, v in zip(bars, state_sorted['total_racialised']):
    ax.text(bar.get_width()+2, bar.get_y()+bar.get_height()/2, str(int(v)), va='center', fontsize=8)

ax = axes[1]
bars2 = ax.barh(state_sorted['LAND_NAME'], state_sorted['n_pools'], color='#55A868', edgecolor='white')
ax.set_xlabel('Number of pools with racialised reviews')
ax.set_title('Pools with racialised reviews per state')
for bar, v in zip(bars2, state_sorted['n_pools']):
    ax.text(bar.get_width()+1, bar.get_y()+bar.get_height()/2, str(int(v)), va='center', fontsize=8)

plt.tight_layout()
plt.savefig('../notebooks/visualisations/state_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Merge constituency counts back to the shapefile for choropleth
wk_plot = wk.merge(
    const_df[['WKR_NR','n_pools','total_racialised']],
    on='WKR_NR', how='left'
).fillna({'n_pools': 0, 'total_racialised': 0})

fig, axes = plt.subplots(1, 2, figsize=(18, 10))
fig.suptitle('Spatial Distribution of Racialised Reviews – Constituency Level', fontsize=14, fontweight='bold')

wk_plot.plot(
    column='total_racialised', ax=axes[0],
    cmap='YlOrRd', legend=True,
    legend_kwds={'label': 'Total racialised reviews', 'shrink': 0.6},
    edgecolor='grey', linewidth=0.3, missing_kwds={'color': 'lightgrey'}
)
axes[0].set_title('Total racialised reviews per constituency')
axes[0].axis('off')

wk_plot.plot(
    column='n_pools', ax=axes[1],
    cmap='Blues', legend=True,
    legend_kwds={'label': 'Pools with racialised reviews', 'shrink': 0.6},
    edgecolor='grey', linewidth=0.3, missing_kwds={'color': 'lightgrey'}
)
axes[1].set_title('Number of pools with racialised reviews per constituency')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('../notebooks/visualisations/constituency_choropleth.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
top20_const = const_df.head(20).sort_values('total_racialised')
colors = plt.cm.tab20.colors
state_list = top20_const['LAND_NAME'].unique().tolist()
state_color = {s: colors[i % 20] for i, s in enumerate(state_list)}
bar_colors = [state_color[s] for s in top20_const['LAND_NAME']]

bars = ax.barh(
    top20_const['WKR_NAME'] + '\n(' + top20_const['LAND_NAME'] + ')',
    top20_const['total_racialised'],
    color=bar_colors, edgecolor='white'
)
ax.set_xlabel('Total racialised reviews')
ax.set_title('Top 20 constituencies by total racialised reviews\n(colour = state)')
for bar, row in zip(bars, top20_const.itertuples()):
    ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
            f'{int(row.total_racialised)}  ({int(row.n_pools)} pools)',
            va='center', fontsize=8)
from matplotlib.patches import Patch
legend_patches = [Patch(color=state_color[s], label=s) for s in state_list]
ax.legend(handles=legend_patches, loc='lower right', fontsize=7, title='State')
plt.tight_layout()
plt.savefig('../notebooks/visualisations/top_constituencies.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 6 · Are high-count pools clustered? East vs West? How far apart?

Focusing on the **top 20 pools** by racialised review count.


In [ ]:
from math import radians, sin, cos, sqrt, atan2

gpkg_path = Path('../notebooks/constituency_aggregated.gpkg')
gpkg_gdf  = gpd.read_file(gpkg_path)
ew_map    = gpkg_gdf.set_index('constituency_number')['east_west'].to_dict()

WK_PATH = Path('/home/ethel/Desktop/swimming_pool_project/btw25_geometrie_wahlkreise_shp'
               '/btw25_geometrie_wahlkreise_shp.shp')
wk2 = gpd.read_file(WK_PATH)

pool_df2 = (
    rac_merged
    .groupby(['place_id','place_name','address','latitude','longitude'])
    .size().reset_index(name='racialised_count')
)
pool_gdf2 = gpd.GeoDataFrame(
    pool_df2,
    geometry=[Point(lon, lat) for lon, lat in zip(pool_df2['longitude'], pool_df2['latitude'])],
    crs='EPSG:4326'
).to_crs(wk2.crs)
pool_j2 = gpd.sjoin(
    pool_gdf2, wk2[['WKR_NR','WKR_NAME','LAND_NAME','geometry']],
    how='left', predicate='within'
)
pool_j2['east_west'] = pool_j2['WKR_NR'].map(ew_map)

top20 = pool_j2.nlargest(20, 'racialised_count').reset_index(drop=True)

print('Top 20 pools – constituency, state, east/west:')
top20[['place_name','racialised_count','WKR_NAME','LAND_NAME','east_west']]


In [ ]:
print('East/West split – top 20 pools:')
print(top20.groupby('east_west').agg(
    pools=('place_name','count'),
    total_reviews=('racialised_count','sum'),
    avg_reviews=('racialised_count','mean')
).round(1))
print()
print('East pools:')
print(top20[top20['east_west']=='east'][['place_name','racialised_count','LAND_NAME','WKR_NAME']].to_string(index=False))
print()
print('NRW cluster (same state, very close geographically):')
print(top20[top20['LAND_NAME']=='Nordrhein-Westfalen'][['place_name','racialised_count','WKR_NAME']].to_string(index=False))


In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat, dlon = radians(lat2-lat1), radians(lon2-lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return round(2 * R * atan2(sqrt(a), sqrt(1-a)), 1)

coords = list(zip(top20['latitude'], top20['longitude']))
dists = []
for i in range(len(top20)):
    for j in range(i+1, len(top20)):
        dists.append({
            'pool_a':  top20.loc[i,'place_name'], 'state_a': top20.loc[i,'LAND_NAME'], 'ew_a': top20.loc[i,'east_west'],
            'pool_b':  top20.loc[j,'place_name'], 'state_b': top20.loc[j,'LAND_NAME'], 'ew_b': top20.loc[j,'east_west'],
            'km': haversine(*coords[i], *coords[j])
        })
dist_df = pd.DataFrame(dists)
dist_df['cross_ew'] = dist_df['ew_a'] != dist_df['ew_b']
dist_df['same_state'] = dist_df['state_a'] == dist_df['state_b']

print('Pairwise distance stats (190 pairs):')
print(dist_df['km'].describe().round(1))
print()
print('10 closest pairs:')
print(dist_df.nsmallest(10,'km')[['pool_a','pool_b','state_a','state_b','km']].to_string(index=False))
print()
print('5 farthest pairs:')
print(dist_df.nlargest(5,'km')[['pool_a','pool_b','state_a','state_b','km']].to_string(index=False))


In [ ]:
print('Average distance – same East/West vs cross East/West:')
print(dist_df.groupby('cross_ew')['km'].agg(['mean','median','min','max']).round(1)
      .rename(index={False:'same E/W', True:'cross E/W'}))
print()
print('Average distance – same state vs cross state:')
print(dist_df.groupby('same_state')['km'].agg(['mean','median','min','max']).round(1)
      .rename(index={False:'cross-state', True:'same state'}))


In [ ]:
# Cluster labels for annotation
cluster_map = {
    'Nordrhein-Westfalen': 'NRW cluster',
    'Brandenburg':         'Brandenburg cluster',
    'Baden-Württemberg':   'BW cluster',
}
top20['cluster'] = top20['LAND_NAME'].map(cluster_map).fillna('Other')

cluster_colors = {
    'NRW cluster':          '#C44E52',
    'Brandenburg cluster':  '#4C72B0',
    'BW cluster':           '#55A868',
    'Other':                '#DD8452',
}

# Reproject wk to WGS84 for the map
wk_wgs = wk2.to_crs('EPSG:4326')
top20_wgs = top20.to_crs('EPSG:4326')

fig, ax = plt.subplots(figsize=(12, 14))
wk_wgs.plot(ax=ax, color='#f0f0f0', edgecolor='#cccccc', linewidth=0.3)

for cluster, grp in top20_wgs.groupby('cluster'):
    grp.plot(ax=ax, color=cluster_colors[cluster], markersize=grp['racialised_count']/2,
             alpha=0.85, label=cluster, zorder=3)

# Annotate each pool
for _, row in top20_wgs.iterrows():
    ax.annotate(
        f"{row['place_name'][:28]}\n({int(row['racialised_count'])} rev.)",
        xy=(row.geometry.x, row.geometry.y),
        xytext=(6, 2), textcoords='offset points',
        fontsize=6.5, color='#333333'
    )

ax.set_title('Top 20 pools by racialised reviews\n'
             '(bubble size = review count, colour = state cluster)',
             fontsize=13, fontweight='bold')
ax.axis('off')
ax.legend(title='Cluster', loc='lower left', fontsize=9)

plt.tight_layout()
plt.savefig('../notebooks/visualisations/top20_pools_map.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Pairwise distance matrix heatmap for top 20
import numpy as np

names = top20['place_name'].str[:30].tolist()
n = len(top20)
mat = np.zeros((n, n))
for _, row in dist_df.iterrows():
    ia = top20[top20['place_name'] == row['pool_a']].index[0]
    ib = top20[top20['place_name'] == row['pool_b']].index[0]
    mat[ia, ib] = row['km']
    mat[ib, ia] = row['km']

# EW border colour
ew_colors = ['#C44E52' if e == 'east' else '#4C72B0' for e in top20['east_west']]

fig, ax = plt.subplots(figsize=(13, 11))
im = ax.imshow(mat, cmap='YlOrRd_r', aspect='auto')
plt.colorbar(im, ax=ax, label='Distance (km)')
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(names, rotation=60, ha='right', fontsize=7.5)
ax.set_yticklabels(names, fontsize=7.5)
for i, c in enumerate(ew_colors):
    ax.get_xticklabels()[i].set_color(c)
    ax.get_yticklabels()[i].set_color(c)
# annotate cells
for i in range(n):
    for j in range(n):
        if i != j:
            ax.text(j, i, f'{int(mat[i,j])}', ha='center', va='center', fontsize=5.5)
ax.set_title('Pairwise distances (km) between top-20 pools\n'
             'Red label = former East Germany, Blue = former West', fontsize=11)
plt.tight_layout()
plt.savefig('../notebooks/visualisations/top20_distance_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
